# 01 — Dataset Exploration

**Purpose:** Inspect the raw TMDB 5000 Movie Dataset before any preprocessing.

**What we check:**
- File shapes and column names
- Data types and missing values
- Duplicates
- How the three tables connect (ratingId, tmdbId)
- Rating scale and distribution
- Genre and keyword formats
- Unique users and movies

**Output:** Findings recorded here and in the README. No data is modified.

## 1. Load Libraries and Data

In [ ]:
# Import pandas for data manipulation
import pandas as pd

# Import json to parse the JSON-formatted columns (genres, keywords, cast, crew)
import json

# Import matplotlib/seaborn for optional visualisation
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Load the three raw CSV files from data/raw/
# - movies: movie metadata (title, genres, keywords, budget, etc.)
# - ratings: user ratings (userId, rating, timestamp)
# - credits: cast and crew information
movies = pd.read_csv('../data/raw/tmdb_movie_dataset.csv')
ratings = pd.read_csv('../data/raw/tmdb_movie_ratings.csv')
credits = pd.read_csv('../data/raw/tmdb_movie_credits.csv')

## 2. Basic Shape and Columns

In [ ]:
# Check the dimensions (rows x columns) of each dataset
# This tells us how many records and features we have
print('Movies shape:', movies.shape)
print('Ratings shape:', ratings.shape)
print('Credits shape:', credits.shape)

In [ ]:
# List all column names for each dataset
# This helps us understand what fields are available
print('=== Movies columns ===')
print(movies.columns.tolist())
print()
print('=== Ratings columns ===')
print(ratings.columns.tolist())
print()
print('=== Credits columns ===')
print(credits.columns.tolist())

## 3. Data Types and Non-Null Counts

In [ ]:
# .info() shows data types and how many non-null values each column has
# This helps identify columns with missing data and type issues
print('=== Movies info ===')
print(movies.info())
print()
print('=== Ratings info ===')
print(ratings.info())
print()
print('=== Credits info ===')
print(credits.info())

## 4. Missing Values

In [ ]:
# Count missing (NaN) values per column for each dataset
# Columns with high missing counts may need special handling
print('=== Movies missing values ===')
print(movies.isnull().sum())
print()
print('=== Ratings missing values ===')
print(ratings.isnull().sum())
print()
print('=== Credits missing values ===')
print(credits.isnull().sum())

## 5. Duplicates

In [ ]:
# Check for fully duplicated rows across all columns
# Also check for duplicate tmdbId in movies (same movie appearing multiple times)
print('=== Full row duplicates ===')
print(f'Movies: {movies.duplicated().sum()}')
print(f'Ratings: {ratings.duplicated().sum()}')
print(f'Credits: {credits.duplicated().sum()}')
print()

# Check if any tmdbId appears more than once in the movies table
dup_tmdb = movies[movies['tmdbId'].duplicated(keep=False)]
print(f'=== Duplicate tmdbId entries ({len(dup_tmdb)} rows) ===')
print(dup_tmdb[['tmdbId', 'title', 'ratingId']].to_string())

## 6. Table Connections

The three tables are linked by:
- `movies.ratingId` ↔ `ratings.ratingId` (one movie → many user ratings)
- `movies.tmdbId` ↔ `credits.tmdbId` (one-to-one movie metadata)

In [ ]:
# Convert ratingId columns to sets for easy comparison
movies_rid = set(movies['ratingId'])
ratings_rid = set(ratings['ratingId'])

# Check overlap between movies.ratingId and ratings.ratingId
print(f'ratingId in movies:     {len(movies_rid)}')
print(f'ratingId in ratings:    {len(ratings_rid)}')
print(f'In BOTH (joinable):     {len(movies_rid & ratings_rid)}')
print(f'In movies only (no ratings): {len(movies_rid - ratings_rid)}')
print(f'In ratings only (orphan):     {len(ratings_rid - movies_rid)}')

In [ ]:
# Show the movies that have NO corresponding ratings
# These will need to be dropped during preprocessing
no_ratings = movies[~movies['ratingId'].isin(ratings_rid)]
print(f'Movies with no ratings: {len(no_ratings)}')
print(no_ratings[['tmdbId', 'title', 'ratingId']])

In [ ]:
# Check tmdbId overlap between movies and credits
movies_tid = set(movies['tmdbId'])
credits_tid = set(credits['tmdbId'])

print(f'tmdbId in movies:  {len(movies_tid)}')
print(f'tmdbId in credits: {len(credits_tid)}')
print(f'In BOTH:           {len(movies_tid & credits_tid)}')
print(f'In movies only:    {len(movies_tid - credits_tid)}')
print(f'In credits only:   {len(credits_tid - movies_tid)}')

## 7. Rating Analysis

In [ ]:
# Describe the rating column to get min, max, mean, std, quartiles
# This tells us the rating scale and central tendency
print('=== Rating statistics ===')
print(ratings['rating'].describe())
print()
print(f'Unique rating values: {sorted(ratings["rating"].unique())}')

In [ ]:
# Count how many ratings exist for each rating value (0.5, 1.0, ... 5.0)
# This reveals if ratings are skewed (e.g., most ratings are 4.0)
print('=== Rating distribution ===')
rating_dist = ratings['rating'].value_counts().sort_index()
print(rating_dist)

In [ ]:
# Visualise the rating distribution as a bar chart
plt.figure(figsize=(8, 4))
rating_dist.plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Number of Ratings')
plt.tight_layout()
plt.show()

In [ ]:
# Count unique users and unique movies in the ratings table
# This tells us the size of the user-item matrix
print(f'Unique users:   {ratings["userId"].nunique():,}')
print(f'Unique movies:  {ratings["ratingId"].nunique():,}')
print(f'Total ratings:  {len(ratings):,}')
print(f'Sparsity:       {1 - len(ratings) / (ratings["userId"].nunique() * ratings["ratingId"].nunique()):.4%}')

In [ ]:
# Check how many ratings each movie has (top and bottom)
# Identifies popular vs. obscure movies
ratings_per_movie = ratings.groupby('ratingId').size().sort_values(ascending=False)

print('=== Top 10 most-rated movies ===')
top10 = ratings_per_movie.head(10).reset_index()
top10 = top10.merge(movies[['ratingId', 'title']], on='ratingId')
print(top10.to_string(index=False))
print()

print('=== Ratings per movie stats ===')
print(f'Min:    {ratings_per_movie.min()}')
print(f'Max:    {ratings_per_movie.max()}')
print(f'Mean:   {ratings_per_movie.mean():.0f}')
print(f'Median: {ratings_per_movie.median():.0f}')

## 8. Genre and Keyword Formats

In [ ]:
# The 'genres' column is stored as a JSON string, e.g.:
#   [{"id": 35, "name": "Comedy"}, {"id": 80, "name": "Crime"}]
# We parse it to extract the genre names.

all_genres = set()
for g in movies['genres']:
    try:
        for item in json.loads(g):
            all_genres.add(item['name'])
    except (json.JSONDecodeError, TypeError):
        pass  # skip malformed entries

print(f'Total unique genres: {len(all_genres)}')
print(f'Genres: {sorted(all_genres)}')

In [ ]:
# Similarly, parse the 'keywords' column to see what keywords exist
# Keywords are also JSON: [{"id": 612, "name": "hotel"}, ...]

all_keywords = set()
for k in movies['keywords']:
    try:
        for item in json.loads(k):
            all_keywords.add(item['name'])
    except (json.JSONDecodeError, TypeError):
        pass

print(f'Total unique keywords: {len(all_keywords)}')
print(f'First 30 keywords: {sorted(all_keywords)[:30]}')

In [ ]:
# Show how many movies belong to each genre
genre_counts = {}
for g in movies['genres']:
    try:
        for item in json.loads(g):
            name = item['name']
            genre_counts[name] = genre_counts.get(name, 0) + 1
    except (json.JSONDecodeError, TypeError):
        pass

genre_series = pd.Series(genre_counts).sort_values(ascending=False)
print('=== Movies per genre ===')
print(genre_series)

In [ ]:
# Visualise genre counts
plt.figure(figsize=(10, 5))
genre_series.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Number of Movies per Genre')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 9. Sample Data Preview

In [ ]:
# Preview the first few rows of each dataset to see real values
print('=== Movies (first 3 rows, key columns) ===')
print(movies[['tmdbId', 'title', 'genres', 'keywords', 'vote_average', 'ratingId']].head(3).to_string())
print()

print('=== Ratings (first 5 rows) ===')
print(ratings.head().to_string())
print()

print('=== Credits (first 3 rows, key columns) ===')
print(credits[['tmdbId', 'title']].head(3).to_string())

## 10. Summary of Findings

| Property | Movies | Ratings | Credits |
|---|---|---|---|
| Rows | 4,602 | 17,258,140 | 4,602 |
| Columns | 21 | 4 | 4 |
| Missing values | homepage (2,944), tagline (727), overview (1) | None | None |
| Duplicates | 0 | 0 | 4 (duplicate tmdbId) |
| Unique IDs | tmdbId: 4,598 / ratingId: 4,602 | userId: 162,532 / ratingId: 4,595 | tmdbId: 4,598 |

**Key observations:**
1. `ratingId` is the link between movies and ratings (not `tmdbId`).
2. `tmdbId` links movies to credits.
3. 4 movies have duplicate `tmdbId` entries (appear twice with different `ratingId`).
4. 7 movies have no ratings at all (need to be dropped).
5. Rating scale: 0.5 to 5.0 in 0.5 steps.
6. 20 unique genres available for mood mapping.
7. Genres and keywords are stored as JSON strings — need parsing.
8. High sparsity (many users, many movies, but each user rates only a tiny fraction).